# 10 — Hyperparameter Optimization (Optuna)

**Objective:** Systematically tune key hyperparameters using Optuna's Tree-structured Parzen Estimator (TPE) strictly across within-development temporal folds.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.optimization.tuner import TemporalOptimizer
from src.data.loader import FEATURES_A, TARGETS_A
from src.data.splitting import temporal_split


D:\TURBINE_X_Full_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Running Optuna Hyperparameter Optimization


In [2]:
df = pd.read_parquet(ROOT / "data" / "processed" / "cleaned_master.parquet")
dev, cal, test = temporal_split(df)

optimizer = TemporalOptimizer(
    dev_df=dev,
    features=FEATURES_A,
    targets=TARGETS_A,
    n_trials=8,
    random_state=42
)

res = optimizer.optimize_model("HistGradientBoosting")
print(f"Best Optuna RMSE: {res['best_rmse']:.4f}")
print("Best Hyperparameters:")
for k, v in res["best_params"].items():
    print(f"  {k:20s}: {v}")

trials_df = res["trials_df"]
out_trials = ROOT / "experiments" / "optuna_trials.csv"
trials_df.to_csv(out_trials, index=False)
print(f"Saved Optuna trials history to {out_trials.relative_to(ROOT)}")
print("Hyperparameter optimization COMPLETE.")


Best Optuna RMSE: 3.9659
Best Hyperparameters:
  max_iter            : 200
  learning_rate       : 0.17254716573280354
  max_leaf_nodes      : 50
  l2_regularization   : 0.24810409748678125
Saved Optuna trials history to experiments\optuna_trials.csv
Hyperparameter optimization COMPLETE.
